In [123]:
!pip install pandas scikit-learn

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [124]:
import pandas as pd

# ==========================================
# โหลดข้อมูล
# ==========================================

df = pd.read_csv("dataset.csv")

# แสดงข้อมูล 5 แถวแรก
print(df.head())

# ตรวจสอบจำนวนข้อมูลในแต่ละหมวดหมู่
print("\nจำนวนข้อมูลแต่ละ Category")
print(df["label"].value_counts())

   id                                            text  label  \
0   1   ตรงสนามหญ้าหน้าคณะมีสภาพ ขยะล้น รบกวนดูแลด้วย      0   
1   2      แจ้งเรื่องครับ พอดีเจอว่า แอร์ ที่ตึกวิทย์      1   
2   3     รบกวนตรวจสอบบริเวณประตู 2 มีปัญหา รถชน ครับ      2   
3   4  ขออนุญาตสอบถามเรื่อง ขอสอบชดเชย ของเทอมนี้ครับ      3   
4   5       แจ้งเรื่องครับ พอดีเจอว่า พัง ที่ตึกวิทย์      1   

                         category  
0       หมวดภูมิทัศน์และความสะอาด  
1  หมวดอาคารและสิ่งอำนวยความสะดวก  
2         หมวดความปลอดภัยและจราจร  
3    หมวดการเรียนการสอนและวิชาการ  
4  หมวดอาคารและสิ่งอำนวยความสะดวก  

จำนวนข้อมูลแต่ละ Category
label
1    232
3    221
0    218
4    217
6    214
5    200
2    198
Name: count, dtype: int64


In [125]:
from sklearn.model_selection import train_test_split

# ==========================================
# แบ่งข้อมูลรอบแรก
# 85% สำหรับ Train + Validation
# 15% สำหรับ Test
# ==========================================

train_val, test = train_test_split(
    df,
    test_size=0.15,
    stratify=df["label"],
    random_state=42
)

# ==========================================
# แบ่งข้อมูลที่เหลือ
# ให้เป็น Train และ Validation
# ==========================================

train, validation = train_test_split(
    train_val,
    test_size=0.1765,
    stratify=train_val["label"],
    random_state=42
)

# ==========================================
# บันทึกไฟล์
# ==========================================

train.to_csv("train.csv", index=False)
validation.to_csv("validation.csv", index=False)
test.to_csv("test.csv", index=False)

print("Train:", len(train))
print("Validation:", len(validation))
print("Test:", len(test))

Train: 1049
Validation: 226
Test: 225


In [126]:
def create_typhoon_prompt(raw_text):
    """
    สร้าง Prompt สำหรับให้ Typhoon
    ทำหน้าที่จัดระเบียบข้อมูลจากข้อความแจ้งปัญหา
    """

    prompt = f"""
คุณเป็นระบบช่วยวิเคราะห์ข้อความแจ้งปัญหาภายในมหาวิทยาลัย

หน้าที่ของคุณคือ:
1. สรุปว่าผู้ใช้กำลังแจ้งปัญหาอะไร
2. ดึงสถานที่ ถ้ามี
3. ดึงวันที่เกิดเหตุ ถ้ามี
4. ดึงเวลาเกิดเหตุ ถ้ามี
5. ห้ามแต่งข้อมูลที่ไม่มีในข้อความ
6. ห้ามเปลี่ยนความหมายของข้อความ

ตอบกลับในรูปแบบ JSON เท่านั้น

ข้อความ:
{raw_text}

รูปแบบคำตอบ:

{{
    "problem_summary": "",
    "problem": "",
    "location": null,
    "incident_date": null,
    "incident_time": null
}}
"""

    return prompt

In [127]:
text = "รถเมล์ไม่มาตั้งแต่ 8 โมง รอนานจนเข้าเรียนสาย"

prompt = create_typhoon_prompt(text)

print(prompt)


คุณเป็นระบบช่วยวิเคราะห์ข้อความแจ้งปัญหาภายในมหาวิทยาลัย

หน้าที่ของคุณคือ:
1. สรุปว่าผู้ใช้กำลังแจ้งปัญหาอะไร
2. ดึงสถานที่ ถ้ามี
3. ดึงวันที่เกิดเหตุ ถ้ามี
4. ดึงเวลาเกิดเหตุ ถ้ามี
5. ห้ามแต่งข้อมูลที่ไม่มีในข้อความ
6. ห้ามเปลี่ยนความหมายของข้อความ

ตอบกลับในรูปแบบ JSON เท่านั้น

ข้อความ:
รถเมล์ไม่มาตั้งแต่ 8 โมง รอนานจนเข้าเรียนสาย

รูปแบบคำตอบ:

{
    "problem_summary": "",
    "problem": "",
    "location": null,
    "incident_date": null,
    "incident_time": null
}



In [128]:
def field_accuracy(true_values, predicted_values):
    """
    คำนวณความถูกต้องของข้อมูลแต่ละ Field
    """

    correct = 0

    for true, pred in zip(true_values, predicted_values):

        if str(true).strip() == str(pred).strip():
            correct += 1

    return correct / len(true_values)

#ขั้นที่ 2: WangchanBERTa ช่วยสร้าง 

In [129]:
!pip install -q transformers datasets accelerate scikit-learn sentencepiece


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [130]:
import pandas as pd
import numpy as np

from datasets import Dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

import torch

In [131]:
df = pd.read_csv("dataset.csv")

print("จำนวนข้อมูลทั้งหมด:", len(df))

df.head()

จำนวนข้อมูลทั้งหมด: 1500


,id,text,label,category
0,1,ตรงสนามหญ้าหน้าคณะมีสภาพ ขยะล้น รบกวนดูแลด้วย,0,หมวดภูมิทัศน์และความสะอาด
1,2,แจ้งเรื่องครับ พอดีเจอว่า แอร์ ที่ตึกวิทย์,1,หมวดอาคารและสิ่งอำนวยความสะดวก
2,3,รบกวนตรวจสอบบริเวณประตู 2 มีปัญหา รถชน ครับ,2,หมวดความปลอดภัยและจราจร
3,4,ขออนุญาตสอบถามเรื่อง ขอสอบชดเชย ของเทอมนี้ครับ,3,หมวดการเรียนการสอนและวิชาการ
4,5,แจ้งเรื่องครับ พอดีเจอว่า พัง ที่ตึกวิทย์,1,หมวดอาคารและสิ่งอำนวยความสะดวก


In [132]:
print(df["category"].value_counts())

print("\nจำนวน Label:")
print(df["label"].value_counts().sort_index())

category
หมวดอาคารและสิ่งอำนวยความสะดวก    232
หมวดการเรียนการสอนและวิชาการ      221
หมวดภูมิทัศน์และความสะอาด         218
หมวดบริการทั่วไป / อื่นๆ          217
หมวดเทคโนโลยีสารสนเทศ             214
หมวดการเดินทางและระบบขนส่ง        200
หมวดความปลอดภัยและจราจร           198
Name: count, dtype: int64

จำนวน Label:
label
0    218
1    232
2    198
3    221
4    217
5    200
6    214
Name: count, dtype: int64


In [149]:
categories = [
    "หมวดภูมิทัศน์และความสะอาด",      # 0
    "หมวดอาคารและสิ่งอำนวยความสะดวก",  # 1
    "หมวดความปลอดภัยและจราจร",         # 2
    "หมวดการเรียนการสอนและวิชาการ",     # 3
    "หมวดบริการทั่วไป / อื่นๆ",        # 4
    "หมวดการเดินทางและระบบขนส่ง",      # 5
    "หมวดเทคโนโลยีสารสนเทศ",          # 6
]


# เลข → ชื่อหมวดหมู่
id2label = {
    i: category
    for i, category in enumerate(categories)
}

# ชื่อหมวดหมู่ → เลข
label2id = {
    category: i
    for i, category in enumerate(categories)
}

print("Mapping ของหมวดหมู่")

for i in range(len(categories)):
    print(f"{i} = {id2label[i]}")

Mapping ของหมวดหมู่
0 = หมวดภูมิทัศน์และความสะอาด
1 = หมวดอาคารและสิ่งอำนวยความสะดวก
2 = หมวดความปลอดภัยและจราจร
3 = หมวดการเรียนการสอนและวิชาการ
4 = หมวดบริการทั่วไป / อื่นๆ
5 = หมวดการเดินทางและระบบขนส่ง
6 = หมวดเทคโนโลยีสารสนเทศ


In [150]:
# ==========================================
# รอบที่ 1
# แบ่ง Train 70%
# เหลือ Validation + Test 30%
# ==========================================

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["label"]
)


# ==========================================
# รอบที่ 2
# แบ่ง 30% ที่เหลือออกเป็น
# Validation 15%
# Test 15%
# ==========================================

validation_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)


# ==========================================
# ตรวจสอบจำนวนข้อมูล
# ==========================================

print("จำนวนข้อมูลทั้งหมด:", len(df))

print("\nTrain:", len(train_df))
print("Validation:", len(validation_df))
print("Test:", len(test_df))

จำนวนข้อมูลทั้งหมด: 1500

Train: 1050
Validation: 225
Test: 225


In [151]:
print("TRAIN")
print(train_df["label"].value_counts().sort_index())

print("\nVALIDATION")
print(validation_df["label"].value_counts().sort_index())

print("\nTEST")
print(test_df["label"].value_counts().sort_index())

TRAIN
label
0    153
1    162
2    138
3    155
4    152
5    140
6    150
Name: count, dtype: int64

VALIDATION
label
0    32
1    35
2    30
3    33
4    33
5    30
6    32
Name: count, dtype: int64

TEST
label
0    33
1    35
2    30
3    33
4    32
5    30
6    32
Name: count, dtype: int64


In [152]:
# แปลง Train DataFrame
train_dataset = Dataset.from_pandas(
    train_df[["text", "label"]],
    preserve_index=False
)


# แปลง Validation DataFrame
validation_dataset = Dataset.from_pandas(
    validation_df[["text", "label"]],
    preserve_index=False
)


# แปลง Test DataFrame
test_dataset = Dataset.from_pandas(
    test_df[["text", "label"]],
    preserve_index=False
)


print(train_dataset)
print(validation_dataset)
print(test_dataset)

Dataset({
    features: ['text', 'label'],
    num_rows: 1050
})
Dataset({
    features: ['text', 'label'],
    num_rows: 225
})
Dataset({
    features: ['text', 'label'],
    num_rows: 225
})


In [154]:
#  แก้ num_labels ให้ตรงกับจำนวนหมวดจริง
model = AutoModelForSequenceClassification.from_pretrained(
    "airesearch/wangchanberta-base-att-spm-uncased",
    num_labels=7,                              # ✅ เปลี่ยนจาก 8 → 7
    problem_type="single_label_classification",
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

print("โหลด WangchanBERTa สำเร็จ")
print("จำนวนหมวดหมู่:", model.config.num_labels)  # ควรได้ 7


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

CamembertForSequenceClassification LOAD REPORT from: airesearch/wangchanberta-base-att-spm-uncased
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


โหลด WangchanBERTa สำเร็จ
จำนวนหมวดหมู่: 7


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("airesearch/wangchanberta-base-att-spm-uncased")
print("Tokenizer loaded")

In [155]:
def tokenize_function(examples):
    
    return tokenizer(
        examples["text"],
        
        # ตัดข้อความหากยาวเกินกำหนด
        truncation=True,
        
        # ความยาวสูงสุด
        max_length=128
    )

In [156]:
train_tokenized = train_dataset.map(
    tokenize_function,
    batched=True
)

validation_tokenized = validation_dataset.map(
    tokenize_function,
    batched=True
)

test_tokenized = test_dataset.map(
    tokenize_function,
    batched=True
)

print(train_tokenized)

Map:   0%|          | 0/1050 [00:00<?, ? examples/s]

Map:   0%|          | 0/225 [00:00<?, ? examples/s]

Map:   0%|          | 0/225 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 1050
})


In [157]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [158]:
def compute_metrics(eval_pred):
    
    # แยก Prediction และ Label จริง
    logits, labels = eval_pred
    
    # เลือกหมวดหมู่ที่มีคะแนนสูงที่สุด
    predictions = np.argmax(
        logits,
        axis=-1
    )
    
    
    # Accuracy
    accuracy = accuracy_score(
        labels,
        predictions
    )
    
    
    # Precision / Recall / F1
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )
    
    
    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [159]:
training_args = TrainingArguments(
    
    # โฟลเดอร์เก็บโมเดล
    output_dir="./wangchanberta_results",
    
    # จำนวนรอบในการฝึก
    num_train_epochs=5,
    
    # จำนวนข้อมูลต่อ Batch
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    
    # Learning Rate
    learning_rate=2e-5,
    
    # Weight Decay
    weight_decay=0.01,
    
    # บันทึก Log
    
    logging_steps=10,
    
    # ไม่ส่งข้อมูลไปบริการภายนอก
    report_to="none"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [160]:
trainer = Trainer(
    
    # โมเดล
    model=model,
    
    # การตั้งค่าการ Train
    args=training_args,
    
    # ข้อมูลสำหรับ Train
    train_dataset=train_tokenized,
    
    # ข้อมูลสำหรับ Validation
    eval_dataset=validation_tokenized,
    
    # จัด Padding
    data_collator=data_collator,
    
    # ฟังก์ชันวัดผล
    compute_metrics=compute_metrics
)

print("สร้าง Trainer สำเร็จ!")

สร้าง Trainer สำเร็จ!


In [161]:
trainer.train()

Step,Training Loss
10,1.866771
20,1.685956
30,1.440573
40,1.199022
50,0.933694
60,0.772879
70,0.506381
80,0.537764
90,0.434374
100,0.291730


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=660, training_loss=0.17973680818227655, metrics={'train_runtime': 58.9065, 'train_samples_per_second': 89.124, 'train_steps_per_second': 11.204, 'total_flos': 43891155461100.0, 'train_loss': 0.17973680818227655, 'epoch': 5.0})

In [162]:
trainer.save_model(
    "./wangchanberta_problem_classifier"
)

tokenizer.save_pretrained(
    "./wangchanberta_problem_classifier"
)

print("บันทึกโมเดลสำเร็จ!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

บันทึกโมเดลสำเร็จ!


In [163]:
test_results = trainer.predict(test_dataset=test_tokenized)
print(test_results.metrics)

{'test_loss': 0.0011768960393965244, 'test_accuracy': 1.0, 'test_precision': 1.0, 'test_recall': 1.0, 'test_f1': 1.0, 'test_runtime': 0.4848, 'test_samples_per_second': 464.138, 'test_steps_per_second': 59.822}


In [164]:
# ให้โมเดลทำนาย Test Set
predictions_output = trainer.predict(
    test_tokenized
)

# คะแนนจากโมเดล
logits = predictions_output.predictions

# เลือกหมวดที่คะแนนสูงที่สุด
predictions = np.argmax(
    logits,
    axis=1
)

# Label จริง
true_labels = predictions_output.label_ids

In [165]:
print(
    classification_report(
        true_labels,
        predictions,
        target_names=categories,
        zero_division=0
    )
)

                                precision    recall  f1-score   support

     หมวดภูมิทัศน์และความสะอาด       1.00      1.00      1.00        33
หมวดอาคารและสิ่งอำนวยความสะดวก       1.00      1.00      1.00        35
       หมวดความปลอดภัยและจราจร       1.00      1.00      1.00        30
  หมวดการเรียนการสอนและวิชาการ       1.00      1.00      1.00        33
      หมวดบริการทั่วไป / อื่นๆ       1.00      1.00      1.00        32
    หมวดการเดินทางและระบบขนส่ง       1.00      1.00      1.00        30
         หมวดเทคโนโลยีสารสนเทศ       1.00      1.00      1.00        32

                      accuracy                           1.00       225
                     macro avg       1.00      1.00      1.00       225
                  weighted avg       1.00      1.00      1.00       225



In [166]:
cm = confusion_matrix(
    true_labels,
    predictions
)

cm

array([[33,  0,  0,  0,  0,  0,  0],
       [ 0, 35,  0,  0,  0,  0,  0],
       [ 0,  0, 30,  0,  0,  0,  0],
       [ 0,  0,  0, 33,  0,  0,  0],
       [ 0,  0,  0,  0, 32,  0,  0],
       [ 0,  0,  0,  0,  0, 30,  0],
       [ 0,  0,  0,  0,  0,  0, 32]], dtype=int64)

In [167]:
cm_df = pd.DataFrame(
    cm,
    index=categories,
    columns=categories
)

print(cm_df)

                                หมวดภูมิทัศน์และความสะอาด  \
หมวดภูมิทัศน์และความสะอาด                              33   
หมวดอาคารและสิ่งอำนวยความสะดวก                          0   
หมวดความปลอดภัยและจราจร                                 0   
หมวดการเรียนการสอนและวิชาการ                            0   
หมวดบริการทั่วไป / อื่นๆ                                0   
หมวดการเดินทางและระบบขนส่ง                              0   
หมวดเทคโนโลยีสารสนเทศ                                   0   

                                หมวดอาคารและสิ่งอำนวยความสะดวก  \
หมวดภูมิทัศน์และความสะอาด                                    0   
หมวดอาคารและสิ่งอำนวยความสะดวก                              35   
หมวดความปลอดภัยและจราจร                                      0   
หมวดการเรียนการสอนและวิชาการ                                 0   
หมวดบริการทั่วไป / อื่นๆ                                     0   
หมวดการเดินทางและระบบขนส่ง                                   0   
หมวดเทคโนโลยีสารสนเทศ                            

In [168]:
def predict_problem(text):
    # Tokenize
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    
    #  ย้าย inputs ไปยัง device เดียวกับโมเดล
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # ให้โมเดลทำนาย
    with torch.no_grad():
        outputs = model(**inputs)
    # ดึงคะแนน
    logits = outputs.logits
    
    
    # แปลงเป็น Probability
    probabilities = torch.sigmoid(
    logits
    )[0]
    
    
    # เรียงคะแนนจากมากไปน้อย
    sorted_indices = torch.argsort(
        probabilities,
        descending=True
    )
    
    
    print("ข้อความ:")
    print(text)
    
    print("\nผลการจัดหมวดหมู่:")
    
    for idx in sorted_indices:
        
        label_id = idx.item()
        
        category = id2label[label_id]
        
        probability = probabilities[label_id].item() * 100
        
        print(
            f"{category}: {probability:.2f}%"
        )

In [169]:
predict_problem(
    "วันนี้รอรถเมล์หน้าหอพักนานมากจนเข้าเรียนไม่ทัน"
)

ข้อความ:
วันนี้รอรถเมล์หน้าหอพักนานมากจนเข้าเรียนไม่ทัน

ผลการจัดหมวดหมู่:
หมวดการเดินทางและระบบขนส่ง: 96.45%
หมวดเทคโนโลยีสารสนเทศ: 67.85%
หมวดภูมิทัศน์และความสะอาด: 60.04%
หมวดการเรียนการสอนและวิชาการ: 48.09%
หมวดอาคารและสิ่งอำนวยความสะดวก: 41.59%
หมวดความปลอดภัยและจราจร: 22.00%
หมวดบริการทั่วไป / อื่นๆ: 9.93%


In [170]:
def predict_and_route(text, threshold=0.30):

    # ==========================================
    # 1) ตรวจสอบว่าโมเดลอยู่บน Device ไหน
    # ==========================================

    device = next(model.parameters()).device


    # ==========================================
    # 2) แปลงข้อความเป็น Token
    # ==========================================

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )


    # ==========================================
    # 3) ย้าย Input ไปอยู่ Device เดียวกับ Model
    # ==========================================

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }


    # ==========================================
    # 4) เปลี่ยนโมเดลเป็น Evaluation Mode
    # ==========================================

    model.eval()


    # ==========================================
    # 5) ให้โมเดลทำนาย
    # ==========================================

    with torch.no_grad():

        outputs = model(**inputs)


    # ==========================================
    # 6) ใช้ Sigmoid
    #
    # Multi-label Classification
    # แต่ละหมวดมีคะแนนของตัวเอง
    # ไม่จำเป็นต้องรวมกัน = 100%
    # ==========================================

    probabilities = torch.sigmoid(
        outputs.logits
    )[0]


    # ==========================================
    # 7) แปลงผลลัพธ์กลับมาเป็น CPU
    #
    # เพื่อให้สามารถนำไปใช้ใน Python ได้ง่าย
    # ==========================================

    probabilities = probabilities.cpu()


    # ==========================================
    # 8) เก็บเฉพาะหมวดที่คะแนน
    # มากกว่าหรือเท่ากับ Threshold
    # ==========================================

    results = []

    for i, probability in enumerate(probabilities):

        score = probability.item()

        if score >= threshold:

            results.append({

                "category": id2label[i],

                "probability": round(
                    score * 100,
                    2
                )

            })


    # ==========================================
    # 9) เรียงคะแนนจากมาก → น้อย
    # ==========================================

    results = sorted(
        results,
        key=lambda x: x["probability"],
        reverse=True
    )


    return results

In [172]:
category_to_department = {

    "ปัญหาเกี่ยวกับรถเมล์":
        "หน่วยงานขนส่ง",

    "ปัญหาเกี่ยวกับความสะอาด":
        "หน่วยงานดูแลความสะอาด",

    "ปัญหาสถานที่/อาคาร":
        "หน่วยงานอาคารสถานที่",

    "ปัญหาเกี่ยวกับการเรียน":
        "หน่วยงานด้านวิชาการ",

    "ปัญหาโรงอาหาร":
        "หน่วยงานโรงอาหาร",

    "ความปลอดภัย/จราจร":
        "หน่วยงานรักษาความปลอดภัย",

    "ระบบเครือข่าย/IT":
        "ศูนย์เทคโนโลยีสารสนเทศ",

    "อื่นๆ":
        "ผู้ดูแลระบบ"
}

In [187]:
result = predict_and_route(
    "เข้าเน็ตไม่ได้",
    threshold=0.30
)

for item in result:

    print(
        f"{item['category']} : "
        f"{item['probability']}%"
    )

หมวดเทคโนโลยีสารสนเทศ : 99.17%
หมวดอาคารและสิ่งอำนวยความสะดวก : 46.13%
หมวดการเรียนการสอนและวิชาการ : 40.11%
หมวดภูมิทัศน์และความสะอาด : 38.87%
หมวดบริการทั่วไป / อื่นๆ : 32.39%


In [174]:
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report

# ✅ รองรับภาษาไทย
matplotlib.rcParams['font.family'] = 'TH Sarabun New'  # หรือ 'Tahoma'
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style="whitegrid", palette="muted")
